# 策略研发全流程 Pipeline

本 Notebook 是量化策略研发的统一核心入口，严格遵循 `strategy_rules.md` 中的样本内外划分与无前视偏差规则：
- **IS_Train (滚动训练集)**: `2021-01-01` ~ `2023-09-30`，仅用于 Rolling WFV 内部早停与调参。
- **IS_Test (核心测试集)**: `2023-10-01` ~ `2024-09-01`，使用最近 4-Fold 集成进行验证，决定策略是否可以进入实盘预备。
- **OOS (严格样本外)**: `2024-09-01` 之后，作为盲区进行最后一道防线检验。

In [ ]:
import sys
import logging
from pathlib import Path

# 配置项目根目录
ROOT = Path("/root/autodl-tmp/Strategy")
sys.path.insert(0, str(ROOT.parent))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

from Strategy import config


## 1. 数据准备 (Label & Factors)
在全市场范围内生成所需的 Label 和因子。这里以 `OPEN0935_1000` 标签为例。

In [ ]:
from Strategy.label.label_generator import generate_and_save_open0935_1000_label

LABEL_TAG = "OPEN0935_1000"
# 可选: start_date=20210104, end_date=20251231 缩小范围以加快调试
open_label_path = generate_and_save_open0935_1000_label(save_price_tables=True)
print(f"[{LABEL_TAG}] Label已生成: {open_label_path}")


In [ ]:
from Strategy.factor.daily_factor_library import DailyFactorLibraryAdapter

# 每日频因子需覆盖到当前可用的最新日期
adapter = DailyFactorLibraryAdapter()
saved_daily = adapter.compute_and_save_all(
    start_date=config.IS_TRAIN_START,
    # end_date=config.IS_TRAIN_END,  # 仅测试时取消注释以加速
)
print(f"已保存 {len(saved_daily)} 个日频因子")


In [ ]:
from Strategy.factor.factor_base import FactorRegistry
import Strategy.factor.minute_derived_factors   # 触发注册
import Strategy.factor.custom_factors           # 触发注册

# 计算并保存所有注册的高频特征
FactorRegistry.compute_all()


## 2. 面板构建与样本拆分
严格按照配置，物理隔离出 `is_train`, `is_test`, `oos`。模型训练时将绝不碰触 `is_test` 和 `oos`。

In [ ]:
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel
import pandas as pd

factor_dict = load_all_factors()
label_df = load_label(LABEL_TAG)

# 展平构建 Panel
panel = build_panel(factor_dict, label_df)

# 严格切割样本 (依据 config 中定好的常数)
is_train, is_test, oos = split_panel(panel)

print(f"IS_Train: {is_train.shape[0]} rows (截止 {is_train['TRADE_DATE'].max().date()})")
print(f"IS_Test:  {is_test.shape[0]} rows (截止 {is_test['TRADE_DATE'].max().date()})")
print(f"OOS:      {oos.shape[0]} rows (起始 {oos['TRADE_DATE'].min().date()})")


## 3. IS Train 滚动训练 (Rolling CV)
利用 `is_train` 内部通过时间窗滚动的方式进行训练和早停，并保存模型参数。

In [ ]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={
        "use_wpcc": True, 
        "label_winsorize_sigma": 3.0
    }
)

# 仅在 IS_Train 上执行所有 Fold 的滚动训练
rt_xgb.train_all_folds(is_train)

# 打印内部交叉验证 IC
print("\n========== XGB CV Fold IC ==========")
print(rt_xgb.fold_ic_report())

# 保存滚动模型结构
rt_xgb.save_model(config.SCORE_OUTPUT_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")


In [ ]:
from Strategy.model.transformer_trainer import TransformerTrainer

rt_tf = RollingTrainer(
    model_class=TransformerTrainer,
    model_kwargs={
        "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
        "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 16,
        "early_stopping_patience": 8, "device": "cuda", "use_amp": False,
        "label_winsorize_sigma": 3.0,
    }
)

rt_tf.train_all_folds(is_train)

print("\n========== Transformer CV Fold IC ==========")
print(rt_tf.fold_ic_report())
rt_tf.save_model(config.SCORE_OUTPUT_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")


## 4. IS Test 核心推理与模型集成
利用最近的 4-Fold 模型对 `is_test` 进行打分，随后进行跨模型种类（XGB vs Transformer）的集成。

In [ ]:
from Strategy.model.scorer import mask_scores_by_current_price
from Strategy.data_io.saver import save_wide_table
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

# XGB 推理
score_test_xgb = rt_xgb.predict_is_test(is_test, normalize=True)
score_test_xgb = mask_scores_by_current_price(score_test_xgb, label_tag=LABEL_TAG)
save_wide_table(score_test_xgb, config.SCORE_OUTPUT_DIR / f"SCORE_xgb_{LABEL_TAG}_is_test.fea")

# Transformer 推理
score_test_tf = rt_tf.predict_is_test(is_test, normalize=True)
score_test_tf = mask_scores_by_current_price(score_test_tf, label_tag=LABEL_TAG)
save_wide_table(score_test_tf, config.SCORE_OUTPUT_DIR / f"SCORE_transformer_{LABEL_TAG}_is_test.fea")

# 集成逻辑
score_dfs = {
    "xgb": score_test_xgb,
    "transformer": score_test_tf
}

corr = compute_score_correlation(score_dfs)
print("\n模型打分截面相关性矩阵:")
print(corr)

# 可根据需要挑选相关性不高、IC皆不错的模型进行平均
selected = select_ensemble_models(score_dfs, max_pairwise_corr=0.8)

score_test_ens = ensemble_scores(
    score_dfs,
    selected_models=selected,
    label_tag=f"{LABEL_TAG}_is_test",
    save=True,
    output_dir=config.SCORE_OUTPUT_DIR
)


## 5. 快速业绩回测 (仅观测 IS_Test)
在没有发生前视偏差的数据区间上观测策略真实的盈利能力。

In [ ]:
from Strategy.backtest.quick_backtest import run_quick_backtest

models_to_test = {
    "xgb": score_test_xgb,
    "transformer": score_test_tf,
    "ensemble": score_test_ens
}

for name, score_df in models_to_test.items():
    print(f"\n{'='*50}\n回测评估: {name} (IS_Test)\n{'='*50}")
    
    run_quick_backtest(
        score_df=score_df,
        label_df=label_df,
        n_groups=20,
        output_dir=config.BT_RESULT_DIR / "is_test" / name,
        start_date=config.IS_TEST_START,
        top_ks=(20, 50, 100),
        tail_ks=(20, 50, 100),
        run_ic=True,
        title=f"{name.upper()} | {LABEL_TAG} | IS_Test"
    )


## 6. OOS 盲区验证 (严格门控)
⚠️ **重要提示**：此部分代码仅当上方 `IS_Test` 回测结果各项指标完全达到预期目标，决定实盘部署时，才允许运行！一旦针对 OOS 数据调整超参，OOS 就被“污染”了。

In [ ]:
# score_oos_xgb = rt_xgb.predict_is_test(oos, normalize=True)
# score_oos_tf = rt_tf.predict_is_test(oos, normalize=True)
# score_oos_xgb = mask_scores_by_current_price(score_oos_xgb, label_tag=LABEL_TAG)
# score_oos_tf = mask_scores_by_current_price(score_oos_tf, label_tag=LABEL_TAG)

# # 跨模型集成
# score_oos_ens = ensemble_scores(
#     {"xgb": score_oos_xgb, "transformer": score_oos_tf},
#     selected_models=selected,
#     label_tag=f"{LABEL_TAG}_oos",
#     save=True,
#     output_dir=config.SCORE_OUTPUT_DIR
# )

# # 回测
# run_quick_backtest(
#     score_df=score_oos_ens,
#     label_df=label_df,
#     n_groups=20,
#     output_dir=config.BT_RESULT_DIR / "oos" / "ensemble",
#     start_date=config.OOS_START,
#     top_ks=(20, 50, 100),
#     tail_ks=(20, 50, 100),
#     run_ic=True,
#     title=f"ENSEMBLE | {LABEL_TAG} | OOS"
# )
